# Advanced Python: Hashing and Equality — 30+ Worked Problems

**Format:** one executable Jupyter notebook, with problem statements followed by detailed explanations, reference solutions, and executable assertions. **Audience:** intermediate → advanced Python. **Requirements:** Python 3.10+; Python standard library only. Run **Kernel → Restart & Run All** from top to bottom. The notebook does not depend on outputs, address values, or a particular hash seed.

## Learning goals

- Reason about identity, value equality, hashability, and dictionary/set lookup.
- Uphold the one-way contract: `a == b` **implies** `hash(a) == hash(b)`; collisions are permitted.
- Distinguish `__hash__ = None` from an implementation that raises `TypeError`.
- Handle mutable state, frozen dataclasses, nested collections, inheritance, `NotImplemented`, and cross-type comparisons.
- Build collision-correct keys, normalized keys, cache keys, and domain-specific identifiers.
- Test equality as an equivalence relation and verify hashing invariants.

**How to study:** attempt each problem before expanding your attention to its immediately following solution. Every solution has assertions so failures are visible. Each class is named distinctly to permit running the entire notebook in order. The later capstones put the rules together.

## Contract and common traps

For a useful value type, equality should be **reflexive** (`a == a`), **symmetric**, and **transitive** (special cases such as IEEE NaN deliberately depart from reflexivity). When `a == b`, their hash values must agree. Unequal values *may* share a hash. An object's hash must not change during its lifetime **while it is being used as a dict key or set member**. In practice, prefer immutable value objects.

The dictionary uses the hash to narrow its search and equality to disambiguate collisions. A manually written `__eq__` normally causes Python to assign `__hash__ = None` on that class unless a suitable `__hash__` is explicitly supplied. This is not an invitation to pair value equality with `object.__hash__` (identity hashing).

**Nuance:** a `@property` without a setter only prevents assignment to the public property; it does not freeze `_backing_state`. `@dataclass(frozen=True)` protects fields from ordinary reassignment, not mutable objects reachable *inside* fields and not hostile bypasses such as `object.__setattr__`. Hash randomization means hash values are not portable persistent identifiers. Never assert a specific hash number.

## Part I — Foundations, invariants, and broken keys

### Problem 01 — Identity-based defaults

**Your task.** Create two distinct instances and an alias. Compare identity with equality, and show that instances can be dictionary keys without custom methods. Why must you avoid asserting that two unequal objects have different hashes?

**Solution and reasoning.** The inherited implementations from `object` provide identity-based equality and compatible hashing. Distinct objects are unequal, but hash collisions are legal even for identity hashes. The same object is equal to itself and usable as a key.

In [1]:
class IdentityTicket:
    pass

first = IdentityTicket()
second = IdentityTicket()
alias = first

assert first is alias
assert first == alias
assert first is not second
assert first != second
assert hash(first) == hash(alias)

lookup = {first: "boarding-pass-A", second: "boarding-pass-B"}
assert len(lookup) == 2
assert lookup[alias] == "boarding-pass-A"
print("Default identity semantics: PASS")

Default identity semantics: PASS


### Problem 02 — Overriding equality disables hashing

**Your task.** Implement equality for a mutable value wrapper. Confirm that distinct instances with the same payload compare equal, and inspect both `__hash__` and `collections.abc.Hashable`. Catch the expected error when using it as a dictionary key.

**Solution and reasoning.** Defining `__eq__` without a corresponding hash sets `__hash__ = None` during class creation. Return `NotImplemented` for unrelated types so Python can attempt reflected comparison. Being unhashable is the safe default when the compared value can change.

In [2]:
from collections.abc import Hashable

class MutableScore:
    def __init__(self, points: int):
        self.points = points

    def __eq__(self, other: object):
        if not isinstance(other, MutableScore):
            return NotImplemented
        return self.points == other.points

score_a = MutableScore(7)
score_b = MutableScore(7)
assert score_a is not score_b and score_a == score_b
assert MutableScore.__hash__ is None
assert not isinstance(score_a, Hashable)
try:
    {score_a: "unavailable"}
except TypeError as exc:
    assert "unhashable" in str(exc)
else:
    raise AssertionError("MutableScore should be unhashable")
print("Value equality with safe unhashability: PASS")

Value equality with safe unhashability: PASS


### Problem 03 — Repair equality with a frozen value object

**Your task.** Design an immutable person identifier whose name defines both equality and hashing. Prove that an independently constructed equal identifier retrieves an existing dictionary value; demonstrate frozen assignment protection.

**Solution and reasoning.** A frozen dataclass with `eq=True` (the default) generates compatible value equality and hashing when its fields are themselves hashable. `slots=True` avoids an ordinary instance `__dict__`. Frozen means ordinary assignments are blocked—not an absolute security boundary.

In [3]:
from dataclasses import FrozenInstanceError, dataclass

@dataclass(frozen=True, slots=True)
class PersonId:
    name: str

original = PersonId("Ada")
equivalent = PersonId("Ada")
different = PersonId("Grace")
assert original is not equivalent
assert original == equivalent
assert original != different
assert hash(original) == hash(equivalent)

people = {original: {"role": "engineer"}}
assert people[equivalent]["role"] == "engineer"
assert len({original, equivalent, different}) == 2
try:
    original.name = "Other"
except (FrozenInstanceError, AttributeError):
    pass
else:
    raise AssertionError("Frozen assignment unexpectedly worked")
print("Immutable value object: PASS")

Immutable value object: PASS


### Problem 04 — Diagnose a mutated dictionary key

**Your task.** Construct a deliberately broken key whose hash follows a mutable integer. Insert it into a dictionary, mutate it, and explain why even looking up the *same instance* becomes unreliable. Repair by leaving the original hash-defining value unchanged.

**Solution and reasoning.** A dict records the key's original hash at insertion. Changing a field used in `__hash__` can direct later lookups to a different location. This is a demonstration of an invalid design, not a recommended technique. Integer hashes for these small positive integers make the example reproducible.

In [4]:
class DangerousKey:
    def __init__(self, number: int):
        self.number = number

    def __eq__(self, other: object):
        if not isinstance(other, DangerousKey):
            return NotImplemented
        return self.number == other.number

    def __hash__(self) -> int:
        return hash(self.number)

key = DangerousKey(11)
indexed = {key: "stored"}
assert indexed[key] == "stored"
key.number = 12  # INVALID once the object is used as a key.
assert hash(11) != hash(12)
assert indexed.get(key) is None
assert indexed.get(DangerousKey(12)) is None
key.number = 11
assert indexed[key] == "stored"
print("Mutation hazard reproduced; keep hash-defining state immutable")

Mutation hazard reproduced; keep hash-defining state immutable


### Problem 05 — A read-only property is not immutability

**Your task.** Reproduce the weakness in the supplied lesson's read-only-property approach. Show that reassignment to the public property fails but mutation of the private backing attribute succeeds. Demonstrate an alternative.

**Solution and reasoning.** `property` without a setter protects only `obj.code = ...`; it does not protect `obj._code`. Prefer a frozen value object with immutable fields and avoid mutation after hashing or dictionary insertion.

In [5]:
class PropertyOnly:
    def __init__(self, code: int):
        self._code = code

    @property
    def code(self) -> int:
        return self._code

    def __eq__(self, other: object):
        if not isinstance(other, PropertyOnly):
            return NotImplemented
        return self.code == other.code

    def __hash__(self) -> int:
        return hash(self.code)

weak = PropertyOnly(3)
try:
    weak.code = 4
except AttributeError:
    pass
else:
    raise AssertionError("Property assignment should fail")
weak._code = 4  # Public read-only property does not prevent this.
assert weak.code == 4

@dataclass(frozen=True, slots=True)
class ProtectedCode:
    code: int

safe = ProtectedCode(3)
assert {safe: "ok"}[ProtectedCode(3)] == "ok"
print("Property loophole diagnosed; frozen value-object alternative works")

Property loophole diagnosed; frozen value-object alternative works


### Problem 06 — Choose the correct fields for a composite hash

**Your task.** Build a line-item key comprising order number, SKU, and line number. Show that changing any component changes equality, and that hashes agree for equal instances without requiring all unequal hashes to differ.

**Solution and reasoning.** Hash the same immutable logical fields that determine equality. Generated dataclass methods implement a coherent composite model. Do not assert that unequal objects always have different hash values.

In [6]:
@dataclass(frozen=True, slots=True)
class LineItemKey:
    order_id: int
    sku: str
    line: int

base = LineItemKey(1001, "PEN", 1)
same = LineItemKey(1001, "PEN", 1)
variants = [
    LineItemKey(1002, "PEN", 1),
    LineItemKey(1001, "BOOK", 1),
    LineItemKey(1001, "PEN", 2),
]
assert base == same and hash(base) == hash(same)
assert all(base != item for item in variants)
assert len({base, same, *variants}) == 4
assert {base: "fulfilled"}[same] == "fulfilled"
print("Composite key contract: PASS")

Composite key contract: PASS


### Problem 07 — Exclude display metadata from equality

**Your task.** Represent a resource by its canonical URI but carry a mutable last-seen counter that should not distinguish resources. Exclude metadata using `field(compare=False)` and verify both equal-key lookup and that metadata updates cannot change hashing.

**Solution and reasoning.** Dataclasses normally exclude `compare=False` fields from both generated comparisons and generated hashes. Excluding metadata is safe only when the domain genuinely considers it irrelevant to key identity. Frozen objects can still hold mutable nested members; avoid that where possible.

In [7]:
from dataclasses import field

@dataclass(frozen=True, slots=True)
class ResourceId:
    uri: str
    last_seen_sequence: int = field(default=0, compare=False)

left = ResourceId("urn:demo:alpha", last_seen_sequence=1)
right = ResourceId("urn:demo:alpha", last_seen_sequence=999)
assert left == right
assert hash(left) == hash(right)
records = {left: "resource"}
assert records[right] == "resource"
assert len({left, right}) == 1
print("Explicit equality semantics for metadata: PASS")

Explicit equality semantics for metadata: PASS


### Problem 08 — Frozen is shallow: the list trap

**Your task.** Show why a frozen dataclass containing a list is still unhashable in practice, even though its class has a generated `__hash__`. Then construct a defensive-copying immutable alternative from any iterable.

**Solution and reasoning.** The generated hash tries to hash the list and fails. `frozen=True` prevents assignment to the field, not mutation of the list stored in it. Copy the input into a tuple at construction, so later changes to the caller's list cannot change equality or hashing.

In [8]:
@dataclass(frozen=True)
class BadFrozenBasket:
    items: list[str]

bad = BadFrozenBasket(["apple"])
assert isinstance(bad, Hashable)  # Advertises hashability; actual hash still fails.
try:
    hash(bad)
except TypeError:
    pass
else:
    raise AssertionError("Lists should not be hashable")
bad.items.append("banana")
assert bad.items == ["apple", "banana"]

@dataclass(frozen=True, slots=True, init=False)
class FrozenBasket:
    items: tuple[str, ...]

    def __init__(self, items):
        snapshot = tuple(items)
        if not all(isinstance(item, str) for item in snapshot):
            raise TypeError("items must contain strings")
        object.__setattr__(self, "items", snapshot)

source = ["apple"]
basket = FrozenBasket(source)
source.append("banana")
assert basket.items == ("apple",)
assert {basket: "checkout"}[FrozenBasket(["apple"])] == "checkout"
print("Shallow-frozen trap fixed by defensive copying")

Shallow-frozen trap fixed by defensive copying


### Problem 09 — Nested hashability and tuples

**Your task.** Predict which nested collection structures can be hashed. Convert nested mutable data to immutable tuples, ensuring that later mutations to source lists do not alter the key.

**Solution and reasoning.** An outer tuple is hashable only if each recursively hashed element is hashable. `tuple([list])` does not freeze the nested list. Copy both outer and inner sequences when constructing a stable key.

In [9]:
invalid = ("orders", [1, 2])
try:
    hash(invalid)
except TypeError:
    pass
else:
    raise AssertionError("Tuple with a list should fail")

raw_rows = [[1, 2], [3, 4]]
key_matrix = tuple(tuple(row) for row in raw_rows)
original_hash = hash(key_matrix)
raw_rows[0].append(99)
raw_rows.append([5])
assert key_matrix == ((1, 2), (3, 4))
assert hash(key_matrix) == original_hash
assert {key_matrix: "snapshot"}[((1, 2), (3, 4))] == "snapshot"
print("Recursive immutable snapshot: PASS")

Recursive immutable snapshot: PASS


## Part II — Comparison semantics, types, and inheritance

### Problem 10 — Return NotImplemented for unsupported comparisons

**Your task.** Implement exact-type equality for a key family and preserve hashing across subclasses. Verify comparisons with strings and unrelated objects are unequal, and compare two matching subclass instances.

**Solution and reasoning.** Returning `NotImplemented` delegates to Python's reflected equality machinery; it is not the boolean `False`, and using `bool(NotImplemented)` is wrong. Strict runtime-type checks prevent accidental cross-type equality; include the runtime type in the hash to mirror the model, though distinct types could legally collide.

In [10]:
class StrictKey:
    def __init__(self, value: str):
        self.value = value

    def __eq__(self, other: object):
        if type(self) is not type(other):
            return NotImplemented
        return self.value == other.value

    def __hash__(self) -> int:
        return hash((type(self), self.value))

class SpecialStrictKey(StrictKey):
    pass

x = StrictKey("a")
y = StrictKey("a")
z = SpecialStrictKey("a")
assert x == y and hash(x) == hash(y)
assert x != z
assert z == SpecialStrictKey("a")
assert x != "a"
assert {x: 1}[y] == 1
print("Type-aware comparison and hashing: PASS")

Type-aware comparison and hashing: PASS


### Problem 11 — Expose and fix asymmetric equality

**Your task.** Create two broken classes where `left == right` differs from `right == left`. Then replace them with a single well-defined value type and test symmetry over a sample population.

**Solution and reasoning.** Python permits user-defined comparisons that violate mathematical equivalence; dictionary and set behavior then becomes difficult to reason about. Equality should be symmetric across all participating types. For a strict value type, decline cross-type equality on both sides.

In [11]:
class BrokenLeft:
    def __eq__(self, other: object):
        if isinstance(other, BrokenRight):
            return True
        return self is other

class BrokenRight:
    def __eq__(self, other: object):
        if isinstance(other, BrokenLeft):
            return False
        return self is other

bl, br = BrokenLeft(), BrokenRight()
assert (bl == br) is True
assert (br == bl) is False

@dataclass(frozen=True, slots=True)
class SymmetricId:
    value: int

sample = [SymmetricId(1), SymmetricId(1), SymmetricId(2), "1"]
assert all((a == b) == (b == a) for a in sample for b in sample)
print("Asymmetry detected and repaired")

Asymmetry detected and repaired


### Problem 12 — Approximate equality is not an equivalence relation

**Your task.** Demonstrate nontransitivity of a tolerance-based equality policy with three numbers. Replace it with a discretized bucket key so equality is transitive, and write a hash implementation that matches the new equality.

**Solution and reasoning.** Close-enough comparisons depend on neighboring values: A may be close to B, and B to C, while A is not close to C. This is unsuitable for hash keys. Define a canonical integer bucket first, and compare/hash that bucket. Note that bucket boundaries are a domain decision, not a general replacement for numeric `isclose`.

In [12]:
import math

class AlmostEqual:
    def __init__(self, value: float):
        self.value = value

    def __eq__(self, other: object):
        if not isinstance(other, AlmostEqual):
            return NotImplemented
        return math.isclose(self.value, other.value, abs_tol=0.11, rel_tol=0.0)

left, middle, right = (AlmostEqual(v) for v in (0.0, 0.1, 0.2))
assert left == middle and middle == right and left != right
assert AlmostEqual.__hash__ is None

@dataclass(frozen=True, slots=True)
class BucketKey:
    bucket: int

    @classmethod
    def from_measurement(cls, value: float) -> "BucketKey":
        if not math.isfinite(value):
            raise ValueError("measurement must be finite")
        return cls(math.floor(value * 10))

keys = [BucketKey.from_measurement(v) for v in (0.01, 0.09, 0.11)]
assert keys[0] == keys[1] and keys[1] != keys[2]
assert hash(keys[0]) == hash(keys[1])
print("Nontransitive tolerance replaced with canonical buckets")

Nontransitive tolerance replaced with canonical buckets


### Problem 13 — The bool/int/float equality surprise

**Your task.** Show how Python's numeric equality merges `True`, `1`, and `1.0` in a dictionary. Build a typed wrapper that intentionally distinguishes these three value types while preserving the hash contract.

**Solution and reasoning.** Built-in numeric equality intentionally treats these as equal, and their hashes match accordingly. If an application requires *type-sensitive* keys, the wrapper's equality and hash must both incorporate the exact value type.

In [13]:
assert True == 1 == 1.0
assert hash(True) == hash(1) == hash(1.0)
plain = {True: "bool", 1: "int", 1.0: "float"}
assert len(plain) == 1 and plain[True] == "float"

@dataclass(frozen=True, slots=True, eq=False)
class TypedNumber:
    value: object

    def __eq__(self, other: object):
        if not isinstance(other, TypedNumber):
            return NotImplemented
        return (type(self.value) is type(other.value)
                and self.value == other.value)

    def __hash__(self) -> int:
        return hash((type(self.value), self.value))

typed = {TypedNumber(True): "bool", TypedNumber(1): "int",
         TypedNumber(1.0): "float"}
assert len(typed) == 3
assert typed[TypedNumber(1)] == "int"
print("Intentional type-sensitive key semantics: PASS")

Intentional type-sensitive key semantics: PASS


### Problem 14 — Collisions are correct, not a failure

**Your task.** Force every key to hash to the same integer. Show that dictionaries and sets still distinguish unequal values and deduplicate equal values. Measure comparison calls instead of assuming collision-free hashes.

**Solution and reasoning.** A hash only selects candidate positions; equality resolves collisions. Constant hashes are functionally legal but degrade performance, and must not be used for general-purpose high-volume keys.

In [14]:
class CollidingCode:
    equality_calls = 0

    def __init__(self, code: int):
        self.code = code

    def __eq__(self, other: object):
        type(self).equality_calls += 1
        if not isinstance(other, CollidingCode):
            return NotImplemented
        return self.code == other.code

    def __hash__(self) -> int:
        return 17  # Deliberately pathological.

collision_map = {CollidingCode(i): i * i for i in range(40)}
assert len(collision_map) == 40
assert collision_map[CollidingCode(12)] == 144
assert len({CollidingCode(1), CollidingCode(1), CollidingCode(2)}) == 2
assert CollidingCode.equality_calls > 0
print("Correct under collisions; equality checks observed:", CollidingCode.equality_calls)

Correct under collisions; equality checks observed: 795


### Problem 15 — Hash randomization and persistence

**Your task.** Explain why a built-in string hash must not be stored as a persistent database ID. Verify that hashing is stable during one process and use a deterministic *digest* for persistent content identifiers.

**Solution and reasoning.** Python may randomize string and bytes hashing between interpreter processes; even nonrandomized hashes need not be stable across Python implementations or builds. For persistent identifiers, use a canonical byte encoding and a named digest (e.g., SHA-256). Do not confuse a cryptographic digest with `__hash__`, which returns a machine-sized integer for in-memory tables.

In [15]:
import hashlib

label = "example:document:42"
assert hash(label) == hash(label)

def persistent_digest(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

fingerprint = persistent_digest(label)
assert len(fingerprint) == 64
assert fingerprint == persistent_digest("example:document:42")
assert fingerprint != persistent_digest("example:document:43")
print("Stable content digest example:", fingerprint[:16] + "…")

Stable content digest example: 0f5cd4eb832dfbec…


**Extra challenge / verification.** Run the following independent extension.

In [16]:
# Optional CPython demonstration: independent processes with explicit hash seeds.
# Differences are typical, not a property to rely upon in application tests.
import os
import subprocess
import sys

snippet = "print(hash('stable text?'))"
values = []
for seed in ("1", "2"):
    environment = dict(os.environ, PYTHONHASHSEED=seed)
    value = subprocess.check_output(
        [sys.executable, "-c", snippet], env=environment, text=True
    ).strip()
    values.append(value)
print("Process-specific hashes (seeds 1 and 2):", values)

Process-specific hashes (seeds 1 and 2): ['1120394054584672562', '7400398559672173821']


### Problem 16 — Hash results must be integers

**Your task.** Write a class with an invalid `__hash__` return value. Explain the exception, and contrast it with a valid class whose `__hash__` returns an integer.

**Solution and reasoning.** `__hash__` must return an `int`; Python reduces oversized integers to its native hash width as needed. A class whose hash method raises is not the same as one explicitly marked unhashable with `__hash__ = None`.

In [17]:
class BadHashReturn:
    def __hash__(self):
        return "a-string"

class GoodHashReturn:
    def __hash__(self) -> int:
        return 42

try:
    hash(BadHashReturn())
except TypeError as exc:
    assert "__hash__ method should return an integer" in str(exc)
else:
    raise AssertionError("Non-integer hash result should fail")
assert hash(GoodHashReturn()) == 42
print("Hash return type validated")

Hash return type validated


### Problem 17 — `__hash__ = None` versus raising TypeError

**Your task.** Distinguish a correctly unhashable class from a class that advertises hashability but raises at runtime. Check `collections.abc.Hashable` and `hash()` for each.

**Solution and reasoning.** To explicitly prohibit hashing, set `__hash__ = None`. Implementing a method that raises TypeError instead may make `isinstance(x, Hashable)` report True because the method is present. This is an API design distinction.

In [18]:
class ProperlyUnhashable:
    __hash__ = None

class PretendsHashable:
    def __hash__(self):
        raise TypeError("hash deliberately unavailable")

assert not isinstance(ProperlyUnhashable(), Hashable)
assert isinstance(PretendsHashable(), Hashable)
for candidate in (ProperlyUnhashable(), PretendsHashable()):
    try:
        hash(candidate)
    except TypeError:
        pass
    else:
        raise AssertionError("Hash should fail")
print("Hashable ABC and runtime behavior distinguished")

Hashable ABC and runtime behavior distinguished


### Problem 18 — Subclass overrides can silently disable hashing

**Your task.** Create a hashable base class, then override `__eq__` in a subclass without changing its semantics. Check why the subclass becomes unhashable; explicitly restore the inherited hash only when its comparison contract remains compatible.

**Solution and reasoning.** When `__eq__` is defined in a subclass, Python assigns `__hash__ = None` on that subclass unless it explicitly defines one. `__hash__ = Base.__hash__` is correct **only if** the subclass's equality is consistent with the base hash and its hashed fields remain stable.

In [19]:
class BaseAccountKey:
    def __init__(self, account_id: int):
        self.account_id = account_id

    def __eq__(self, other: object):
        if type(self) is not type(other):
            return NotImplemented
        return self.account_id == other.account_id

    def __hash__(self) -> int:
        return hash((type(self), self.account_id))

class AccidentallyUnhashable(BaseAccountKey):
    def __eq__(self, other: object):
        return super().__eq__(other)

class RepairedAccountKey(BaseAccountKey):
    def __eq__(self, other: object):
        return super().__eq__(other)

    __hash__ = BaseAccountKey.__hash__

assert AccidentallyUnhashable.__hash__ is None
assert not isinstance(AccidentallyUnhashable(1), Hashable)
a, b = RepairedAccountKey(1), RepairedAccountKey(1)
assert a == b and hash(a) == hash(b)
assert {a: "ok"}[b] == "ok"
print("Inherited hash restored under a compatible comparison")

Inherited hash restored under a compatible comparison


### Problem 19 — Dataclass method-generation matrix

**Your task.** Investigate the default behavior of mutable `eq=True`, frozen `eq=True`, and `eq=False` dataclasses. Show why `unsafe_hash=True` is dangerous when equality-relevant fields can change.

**Solution and reasoning.** A typical mutable `@dataclass(eq=True)` is unhashable; `frozen=True, eq=True` generates a hash (provided its fields can be hashed); `eq=False` generally retains identity equality/hash from `object`. `unsafe_hash=True` requests a generated hash without protecting its inputs—use only with a rigorously proven invariant.

In [20]:
@dataclass
class MutableDataclass:
    number: int

@dataclass(frozen=True)
class ImmutableDataclass:
    number: int

@dataclass(eq=False)
class IdentityDataclass:
    number: int

@dataclass(unsafe_hash=True)
class UnsafeDataclass:
    number: int

assert MutableDataclass.__hash__ is None
assert not isinstance(MutableDataclass(1), Hashable)
assert ImmutableDataclass(1) == ImmutableDataclass(1)
assert hash(ImmutableDataclass(1)) == hash(ImmutableDataclass(1))
x, y = IdentityDataclass(1), IdentityDataclass(1)
assert x != y and isinstance(x, Hashable)
unsafe = UnsafeDataclass(1)
previous_hash = hash(unsafe)
unsafe.number = 2
assert hash(unsafe) != previous_hash  # These small int tuple hashes differ.
print("Dataclass equality/hash variants inspected")

Dataclass equality/hash variants inspected


### Problem 20 — Subclass comparison dispatch order

**Your task.** Instrument base and derived `__eq__` methods to observe Python's subclass-priority comparison rule. When a right-hand operand is a strict subclass, which method can run first?

**Solution and reasoning.** Python normally gives the subclass's reflected comparison priority when the operand types differ and the right type subclasses the left type. Use `NotImplemented` when a method does not know how to compare; avoid assuming that the textual left operand always runs first.

In [21]:
dispatch_log = []

class General:
    def __eq__(self, other: object):
        dispatch_log.append("General.__eq__")
        return NotImplemented

class Specialized(General):
    def __eq__(self, other: object):
        dispatch_log.append("Specialized.__eq__")
        if isinstance(other, General):
            return True
        return NotImplemented

parent, child = General(), Specialized()
assert parent == child
action_order = dispatch_log.copy()
assert action_order[0] == "Specialized.__eq__"
print("Observed dispatch order:", action_order)
# This pair is not a recommended domain equality model; it isolates dispatch.

Observed dispatch order: ['Specialized.__eq__']


## Part III — Canonicalization, containers, and API design

### Problem 21 — Unicode normalization and case-insensitive value keys

**Your task.** Implement a user handle where canonically equivalent Unicode spellings and case variants are equal. Store and hash the canonical form, and preserve the original spelling separately for presentation.

**Solution and reasoning.** Canonicalize exactly once during construction; compare/hash the canonical string. NFKC plus casefold is one reasonable *domain-specific* policy, not a universal choice for usernames or security-sensitive identifiers. Validate names, consider confusables separately, and avoid modifying an existing key's normalization rules.

In [22]:
import unicodedata

@dataclass(frozen=True, slots=True, init=False)
class UserHandle:
    canonical: str
    display: str = field(compare=False)

    def __init__(self, raw: str):
        if not isinstance(raw, str) or not raw:
            raise ValueError("handle must be a nonempty string")
        normalized = unicodedata.normalize("NFKC", raw)
        canonical = unicodedata.normalize("NFKC", normalized.casefold())
        object.__setattr__(self, "canonical", canonical)
        object.__setattr__(self, "display", raw)

names = [UserHandle("Straße"), UserHandle("STRASSE"),
         UserHandle("Strasse")]
assert names[0] == names[1] == names[2]
assert len({*names}) == 1
assert {names[0]: "profile"}[names[1]] == "profile"
assert UserHandle("é") == UserHandle("é")
assert names[0].display == "Straße"
print("Canonical Unicode value semantics: PASS")

Canonical Unicode value semantics: PASS


### Problem 22 — Order-independent tag sets

**Your task.** Model a project key with a name and a set of tags. Accept iterable tags, remove duplicates by set semantics, make a defensive immutable snapshot, and reject treating a single string as an iterable of characters.

**Solution and reasoning.** Use `frozenset` when tag order and repetition do not matter, and store a copied immutable collection before allowing hashing. Define normalization of tags separately if desired. A list or ordinary set is not an appropriate hash component.

In [23]:
from collections.abc import Iterable

@dataclass(frozen=True, slots=True, init=False)
class TaggedProject:
    name: str
    tags: frozenset[str]

    def __init__(self, name: str, tags: Iterable[str]):
        if not isinstance(name, str) or not name:
            raise ValueError("name must be a nonempty string")
        if isinstance(tags, (str, bytes)):
            raise TypeError("tags must be an iterable of individual strings")
        snapshot = frozenset(tags)
        if not all(isinstance(tag, str) for tag in snapshot):
            raise TypeError("each tag must be a string")
        object.__setattr__(self, "name", name)
        object.__setattr__(self, "tags", snapshot)

mutable_tags = ["python", "training", "python"]
project = TaggedProject("academy", mutable_tags)
mutable_tags.append("later")
peer = TaggedProject("academy", ["training", "python"])
assert project == peer and hash(project) == hash(peer)
assert project.tags == frozenset({"python", "training"})
assert {project: 9}[peer] == 9
try:
    TaggedProject("academy", "python")
except TypeError:
    pass
else:
    raise AssertionError("A bare string should be rejected")
print("Order-independent, defensive tag key: PASS")

Order-independent, defensive tag key: PASS


### Problem 23 — Identity nodes versus logical-ID nodes in a graph

**Your task.** Implement graph vertices for two domains: one where two distinct nodes must remain distinct even if labels match, and another where equal external IDs represent the same logical vertex. Show the resulting set and dict semantics.

**Solution and reasoning.** Identity semantics suit entities distinguished by instance; default `object` equality/hash is appropriate. For logical value IDs, a frozen dataclass lets independently reconstructed keys retrieve the same adjacency entry. Choose based on the application's meaning, not mere convenience.

In [24]:
class IdentityNode:
    def __init__(self, label: str):
        self.label = label

id_node_a = IdentityNode("server")
id_node_b = IdentityNode("server")
assert id_node_a != id_node_b
assert len({id_node_a, id_node_b}) == 2

@dataclass(frozen=True, slots=True)
class GraphNodeId:
    namespace: str
    external_id: int

logical_a = GraphNodeId("server", 10)
logical_b = GraphNodeId("server", 10)
assert logical_a == logical_b
assert len({logical_a, logical_b}) == 1
adjacency = {logical_a: [GraphNodeId("server", 11)]}
assert adjacency[logical_b] == [GraphNodeId("server", 11)]
print("Identity versus logical-ID graph keys: PASS")

Identity versus logical-ID graph keys: PASS


### Problem 24 — Design safe cache keys for mutable input

**Your task.** Wrap a cached function that expects immutable tuples so callers can pass lists. Prove equivalent inputs hit the same cache entry and that mutating the original list afterward does not mutate an existing cached key.

**Solution and reasoning.** `functools.lru_cache` requires hashable arguments. Convert a list to a tuple at the public boundary. A tuple is a suitable key here because its elements are integers; use recursive conversion if nested mutable inputs are possible. Do not cache directly over mutable list objects.

In [25]:
from functools import lru_cache

calculation_calls = 0

@lru_cache(maxsize=32)
def weighted_total(numbers: tuple[int, ...]) -> int:
    global calculation_calls
    calculation_calls += 1
    return sum((index + 1) * number for index, number in enumerate(numbers))

def cached_weighted_total(numbers) -> int:
    snapshot = tuple(numbers)
    if not all(type(number) is int for number in snapshot):
        raise TypeError("Only integers are supported")
    return weighted_total(snapshot)

inputs = [2, 3, 4]
assert cached_weighted_total(inputs) == 20
assert cached_weighted_total([2, 3, 4]) == 20
assert calculation_calls == 1
inputs.append(5)
assert cached_weighted_total(inputs) == 40
assert calculation_calls == 2
assert weighted_total((2, 3, 4)) == 20
assert calculation_calls == 2
print("Cache stats:", weighted_total.cache_info())

Cache stats: CacheInfo(hits=2, misses=2, maxsize=32, currsize=2)


### Problem 25 — Group records by a normalized domain key

**Your task.** Build a histogram of display names by case-insensitive normalized value. Keep a representative display name while treating spellings of the same canonical name as a single key.

**Solution and reasoning.** Separate canonical *identity* from presentation metadata. Hashable keys should not be mutable presentation records. A dictionary counts canonical objects; use a separate mapping if first-seen display spelling matters.

In [26]:
raw_handles = ["Straße", "STRASSE", "Ada", "ada", "ADA", "Grace"]
counts: dict[UserHandle, int] = {}
first_display: dict[UserHandle, str] = {}

for raw in raw_handles:
    handle = UserHandle(raw)
    counts[handle] = counts.get(handle, 0) + 1
    first_display.setdefault(handle, raw)

assert counts[UserHandle("strasse")] == 2
assert counts[UserHandle("ADA")] == 3
assert counts[UserHandle("grace")] == 1
assert first_display[UserHandle("STRASSE")] == "Straße"
assert len(counts) == 3
print("Canonical counts:", {k.canonical: v for k, v in counts.items()})

Canonical counts: {'strasse': 2, 'ada': 3, 'grace': 1}


### Problem 26 — NaN: an intentional reflexivity exception

**Your task.** Explore floating-point NaN and explain why the same NaN object can still be retrieved from a dictionary, while separately created NaN values are not equal. Do not build ordinary logical identifiers out of raw NaNs.

**Solution and reasoning.** IEEE NaN is unequal to itself. Python containers also have an identity fast-path in relevant membership/equality operations; the *same object* can act as a key even though `nan != nan`. Distinct NaNs may coexist because they compare unequal. For application-level value keys, reject or explicitly canonicalize NaNs.

In [27]:
nan_value = float("nan")
other_nan = float("nan")
assert nan_value != nan_value
assert nan_value != other_nan
nan_lookup = {nan_value: "payload"}
assert nan_lookup[nan_value] == "payload"
assert len({nan_value, nan_value}) == 1
assert len({nan_value, other_nan}) == 2

@dataclass(frozen=True, slots=True, init=False)
class FiniteMeasurement:
    value: float

    def __init__(self, value: float):
        if not math.isfinite(value):
            raise ValueError("NaN and infinity are disallowed")
        object.__setattr__(self, "value", value)

assert FiniteMeasurement(2.5) == FiniteMeasurement(2.5)
try:
    FiniteMeasurement(float("nan"))
except ValueError:
    pass
else:
    raise AssertionError("NaN should be rejected")
print("NaN exception understood and validated")

NaN exception understood and validated


### Problem 27 — Fix a hidden equality/hash mismatch

**Your task.** Debug a class that compares only a customer ID but hashes both customer ID and mutable display name. State two valid repairs: (A) define equality and hashing on only immutable customer ID, or (B) compare/hash all immutable logical fields.

**Solution and reasoning.** The broken implementation can produce equal objects with unequal hashes, violating the contract. Fix A follows logical ID and excludes display metadata; fix B models a complete immutable snapshot. They encode different domain semantics; do not conflate them.

In [28]:
class BrokenCustomer:
    def __init__(self, customer_id: int, display: str):
        self.customer_id = customer_id
        self.display = display

    def __eq__(self, other: object):
        if not isinstance(other, BrokenCustomer):
            return NotImplemented
        return self.customer_id == other.customer_id

    def __hash__(self) -> int:
        return hash((self.customer_id, self.display))  # BUG

bad_a = BrokenCustomer(8, "Alice")
bad_b = BrokenCustomer(8, "Alicia")
assert bad_a == bad_b
# The following is a *contract violation* irrespective of whether a coincidental
# hash collision masks it in a particular interpreter run.

@dataclass(frozen=True, slots=True)
class CustomerById:
    customer_id: int
    display: str = field(compare=False)

@dataclass(frozen=True, slots=True)
class CustomerSnapshot:
    customer_id: int
    display: str

id_a, id_b = CustomerById(8, "Alice"), CustomerById(8, "Alicia")
assert id_a == id_b and hash(id_a) == hash(id_b)
assert {id_a: "customer"}[id_b] == "customer"
assert CustomerSnapshot(8, "Alice") != CustomerSnapshot(8, "Alicia")
print("Two semantically different, consistent repairs demonstrated")

Two semantically different, consistent repairs demonstrated


## Part IV — Advanced tests and capstone designs

### Problem 28 — A reusable contract test harness

**Your task.** Write a finite-population checker that verifies reflexivity, symmetry, transitivity, stable hashes, and equal⇒same-hash. It should detect an intentionally broken pair of classes without relying on unequal hashes being different.

**Solution and reasoning.** A contract checker makes invariants executable and catches broken equality models before keys reach production caches or indexes. These tests assume a normal reflexive value domain (do not include raw NaNs). Test all triples only for small finite samples; larger tests should sample triples instead.

In [29]:
def check_value_contract(values):
    """Return human-readable violations for a small population of hashable values."""
    problems = []
    for index, a in enumerate(values):
        if not (a == a):
            problems.append(f"reflexivity failure at {index}")
        if hash(a) != hash(a):
            problems.append(f"unstable hash at {index}")
        for j, b in enumerate(values):
            if (a == b) != (b == a):
                problems.append(f"symmetry failure at {index}, {j}")
            if a == b and hash(a) != hash(b):
                problems.append(f"hash mismatch at {index}, {j}")
            for k, c in enumerate(values):
                if a == b and b == c and not (a == c):
                    problems.append(f"transitivity failure at {index}, {j}, {k}")
    return problems

valid_population = [LineItemKey(1, "A", 1), LineItemKey(1, "A", 1),
                    LineItemKey(1, "B", 1)]
assert check_value_contract(valid_population) == []
# Asymmetry is detected by a separate comparison-only check because
# BrokenLeft/BrokenRight are intentionally unhashable.
assert (bl == br) != (br == bl)
print("Finite contract checks: PASS")

Finite contract checks: PASS


**Extra challenge / verification.** Run the following independent extension.

In [30]:
# Extend to a deterministic pseudo-random sample; no third-party packages.
import random
rng = random.Random(2026)
random_keys = [LineItemKey(rng.randrange(6), rng.choice(("A", "B")),
                           rng.randrange(3)) for _ in range(24)]
assert check_value_contract(random_keys) == []
print("Randomized contract sample: PASS")

Randomized contract sample: PASS


### Problem 29 — Capstone: defensively copied composite identity

**Your task.** Build a multitenant document key with (1) a normalized tenant, (2) a nonnegative integer document ID, (3) order-independent tags, and (4) order-independent string metadata. Reject invalid inputs and duplicate normalized metadata keys. Prove input mutation cannot change the key or dictionary lookup.

**Solution and reasoning.** This design makes all equivalence decisions during construction, snapshots user-controlled data, stores only recursively hashable types, and lets a frozen dataclass generate matching equality and hashing. Sorting metadata pairs eliminates dictionary insertion-order differences. Validation rejects ambiguous inputs rather than silently changing meaning.

In [31]:
from collections.abc import Mapping

@dataclass(frozen=True, slots=True, init=False)
class DocumentKey:
    tenant: str
    document_id: int
    tags: frozenset[str]
    metadata: tuple[tuple[str, str], ...]

    def __init__(self, tenant: str, document_id: int,
                 tags: Iterable[str], metadata: Mapping[str, str]):
        if not isinstance(tenant, str) or not tenant.strip():
            raise ValueError("tenant must be a nonempty string")
        if type(document_id) is not int or document_id < 0:
            raise ValueError("document_id must be a nonnegative integer, not bool")
        if isinstance(tags, (str, bytes)):
            raise TypeError("tags must be an iterable of strings")
        tag_snapshot = frozenset(tags)
        if not all(isinstance(t, str) for t in tag_snapshot):
            raise TypeError("tag members must be strings")
        if not isinstance(metadata, Mapping):
            raise TypeError("metadata must be a mapping")
        if not all(isinstance(k, str) and isinstance(v, str)
                   for k, v in metadata.items()):
            raise TypeError("metadata keys and values must be strings")
        normalized_items = [(k.casefold(), v) for k, v in metadata.items()]
        if len({k for k, _ in normalized_items}) != len(normalized_items):
            raise ValueError("duplicate metadata keys after normalization")
        object.__setattr__(self, "tenant", tenant.strip().casefold())
        object.__setattr__(self, "document_id", document_id)
        object.__setattr__(self, "tags", tag_snapshot)
        object.__setattr__(self, "metadata", tuple(sorted(normalized_items)))

input_tags = ["release", "api", "release"]
input_metadata = {"Region": "EU", "Owner": "A"}
key_one = DocumentKey("  ACME ", 55, input_tags, input_metadata)
key_two = DocumentKey("acme", 55, ["api", "release"],
                      {"owner": "A", "region": "EU"})
assert key_one == key_two and hash(key_one) == hash(key_two)
index = {key_one: "document-55"}
input_tags.append("changed")
input_metadata["Owner"] = "B"
assert index[key_two] == "document-55"
assert key_one.tags == frozenset({"release", "api"})
assert dict(key_one.metadata) == {"owner": "A", "region": "EU"}
print("Production-style composite identity: PASS")

Production-style composite identity: PASS


**Extra challenge / verification.** Run the following independent extension.

In [32]:
for invalid_args, expected in [
    (("a", True, [], {}), ValueError),
    ((" ", 1, [], {}), ValueError),
    (("a", -1, [], {}), ValueError),
    (("a", 1, "tag", {}), TypeError),
    (("a", 1, [], {"Role": "a", "role": "b"}), ValueError),
    (("a", 1, [], {"count": 7}), TypeError),
]:
    try:
        DocumentKey(*invalid_args)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}: {invalid_args!r}")
print("Capstone validation edge cases: PASS")

Capstone validation edge cases: PASS


### Problem 30 — Capstone: collision-resistant correctness and diagnostics

**Your task.** Implement a small dictionary-backed registry of canonical document keys. Test equal-key overwrite behavior, distinct-key preservation, and deliberate hash collisions using a wrapper. Explain why no program should require unequal objects to have unique hashes.

**Solution and reasoning.** An index must distinguish *logical equality* from *hash uniqueness*. Equal keys replace a dictionary value without adding a slot; unequal keys coexist even under extreme collisions. A registry should accept immutable keys and should never mutate the identity fields after insertion.

In [33]:
class ConstantHashDocument:
    def __init__(self, document: DocumentKey):
        self.document = document

    def __eq__(self, other: object):
        if not isinstance(other, ConstantHashDocument):
            return NotImplemented
        return self.document == other.document

    def __hash__(self) -> int:
        return 0  # Artificial collision stress test.

class DocumentRegistry:
    def __init__(self):
        self._entries: dict[object, str] = {}

    def put(self, key: object, value: str) -> None:
        hash(key)  # Fail early on unhashable keys.
        self._entries[key] = value

    def get(self, key: object) -> str:
        return self._entries[key]

    def __len__(self) -> int:
        return len(self._entries)

registry = DocumentRegistry()
first_key = ConstantHashDocument(DocumentKey("acme", 1, ["a"], {}))
equal_key = ConstantHashDocument(DocumentKey("ACME", 1, ["a"], {}))
second_key = ConstantHashDocument(DocumentKey("acme", 2, ["a"], {}))
registry.put(first_key, "original")
registry.put(equal_key, "updated")
registry.put(second_key, "second")
assert len(registry) == 2
assert registry.get(first_key) == "updated"
assert registry.get(second_key) == "second"
assert hash(first_key) == hash(second_key) and first_key != second_key
print("Registry remains correct under forced collisions: PASS")

Registry remains correct under forced collisions: PASS


## Bonus laboratory — Three tougher diagnostics

Each task combines several rules. The code cells contain complete, runnable reference solutions rather than placeholders.

### Problem 31 — Dictionary overwrite: equality versus key identity

**Your task.** Insert one key, then assign using a different but equal key. Determine dictionary length, value, and which *original key object* appears in iteration. Avoid claiming Python replaces the retained key object on an equal-key overwrite.

**Solution and reasoning.** Assignment to an equal key updates the existing mapping entry's value. The original key object is ordinarily retained by Python's built-in dict. This matters if equal objects have different excluded presentation fields: use the value or a separate metadata store for fresh display state.

In [34]:
original_key = ResourceId("urn:test:1", last_seen_sequence=1)
updated_key = ResourceId("urn:test:1", last_seen_sequence=999)
overwrite_demo = {original_key: "old"}
overwrite_demo[updated_key] = "new"
assert len(overwrite_demo) == 1
assert overwrite_demo[original_key] == "new"
assert next(iter(overwrite_demo)) is original_key
assert next(iter(overwrite_demo)).last_seen_sequence == 1
print("Equal-key overwrite updates value while retaining original key")

Equal-key overwrite updates value while retaining original key


### Problem 32 — Safely reject heterogeneous or invalid key components

**Your task.** Design a validated pair key that accepts *exactly* Python integers (excluding bool), makes order matter, and refuses unhashable elements before trying to create a dictionary entry. Contrast it with a tuple containing mutable lists.

**Solution and reasoning.** Validation should occur at construction boundaries. Because `bool` subclasses `int`, `isinstance(True, int)` is insufficient when boolean IDs are invalid. Tuple-valued keys containing two exact integers have stable hashes, while tuples wrapping lists do not.

In [35]:
@dataclass(frozen=True, slots=True)
class DirectedPair:
    source: int
    destination: int

    def __post_init__(self):
        if type(self.source) is not int or type(self.destination) is not int:
            raise TypeError("endpoints must be exact int values")

forward = DirectedPair(1, 2)
backward = DirectedPair(2, 1)
assert forward != backward
assert {forward: "edge"}[DirectedPair(1, 2)] == "edge"
for invalid in ((True, 2), (1.0, 2), ([1], 2)):
    try:
        DirectedPair(*invalid)
    except TypeError:
        pass
    else:
        raise AssertionError(f"Invalid endpoints accepted: {invalid!r}")
print("Validated directed key: PASS")

Validated directed key: PASS


### Problem 33 — Regression tests for dictionary lookup and hashing

**Your task.** Build a table-driven test for representative keys from earlier problems. For every equal pair, check symmetry, hash agreement, dict retrieval, and set deduplication. Exercise several distinct-key pairs without requiring distinct hashes.

**Solution and reasoning.** This is a practical, reusable regression approach: test what matters to the container protocol rather than hard-coded hash integers or incidental collision patterns. For cross-class pairs, verify equality symmetry separately; leave intentionally broken demo classes out of the positive suite.

In [36]:
valid_equal_pairs = [
    (PersonId("Ada"), PersonId("Ada")),
    (LineItemKey(9, "P", 1), LineItemKey(9, "P", 1)),
    (UserHandle("Straße"), UserHandle("STRASSE")),
    (TaggedProject("demo", ["a", "b"]),
     TaggedProject("demo", ["b", "a"])),
    (DocumentKey("ACME", 2, ["one"], {"Region": "EU"}),
     DocumentKey("acme", 2, ["one"], {"region": "EU"})),
]

for left, right in valid_equal_pairs:
    assert left is not right
    assert left == right and right == left
    assert hash(left) == hash(right)
    assert {left: "works"}[right] == "works"
    assert len({left, right}) == 1

unequal_pairs = [
    (PersonId("Ada"), PersonId("Grace")),
    (LineItemKey(1, "A", 1), LineItemKey(1, "A", 2)),
    (TypedNumber(True), TypedNumber(1)),
    (ConstantHashDocument(DocumentKey("t", 1, [], {})),
     ConstantHashDocument(DocumentKey("t", 2, [], {}))),
]
for left, right in unequal_pairs:
    assert left != right and right != left
    assert len({left, right}) == 2

print(f"Regression suite: {len(valid_equal_pairs)} equal pairs and "
      f"{len(unequal_pairs)} distinct pairs PASS")

Regression suite: 5 equal pairs and 4 distinct pairs PASS


## Review / interview checklist

1. Does the class need identity equality or value equality? Define the domain first.
2. If defining `__eq__`, is `__hash__` intentionally absent (`None`) or a compatible implementation?
3. Are all comparison and hash inputs immutable **in practice**, including nested collections and caller-owned references?
4. Do equal values always have equal hashes? (Do **not** demand the converse.)
5. Are comparisons reflexive, symmetric, and transitive for the intended value domain?
6. Is `NotImplemented` returned for truly unsupported types, and are inheritance/cross-type semantics deliberate?
7. Are `bool` versus `int`, Unicode spelling/casing, ordering of tags, and NaN treated intentionally?
8. Do tests cover collision stress, equal-key retrieval, and equal-key overwrite?
9. Is a persistent identifier implemented with a stable encoding/digest instead of Python's process-specific `hash()`?
10. Can a simpler tuple, `frozenset`, `NamedTuple`, or frozen dataclass replace a custom implementation?

**Final rule:** equality defines the identity of a value; hashing is only a compatible indexing aid. Correctness comes first, performance second.